In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, EarlyStoppingCallback
from datasets import Dataset
import pandas as pd

c:\Users\ADMIN\anaconda3\envs\tracking-barbell-exercises\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\ADMIN\anaconda3\envs\tracking-barbell-exercises\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df_gold = pd.read_csv('data_phase2.csv', encoding = 'utf-8')
new_dataset = Dataset.from_pandas(df_gold[['text', 'target']])
checkpoint_path = r"C:\Code\TeenCodeTranslator\fine_tune\outputs\Trial_FullFT_LR_5e-05\checkpoint-635"


tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path)

def preprocess_function(examples):
    model_inputs = tokenizer(examples["text"], max_length=64, truncation=True)
    labels = tokenizer(text_target=examples["target"], max_length=64, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Chạy map để tokenize data mới
tokenized_new_dataset = new_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=new_dataset.column_names
)

# Chia train/test (90/10) cho Phase 2
split_new_dataset = tokenized_new_dataset.train_test_split(test_size=0.1)

Map: 100%|██████████| 4100/4100 [00:00<00:00, 7309.37 examples/s]


In [4]:
# 1. Gọi tiện ích mở rộng TensorBoard dành riêng cho Jupyter
%load_ext tensorboard

# 2. Khởi chạy giao diện ngay dưới cell
%tensorboard --logdir ./logs

run_name = "Phase2_HardExamples_LR_2e-5"

training_args = Seq2SeqTrainingArguments(
    output_dir=f"./outputs/{run_name}",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    
    # CHIẾN THUẬT 1: Giảm Learning Rate xuống một chút (từ 5e-5 xuống 2e-5) 
    # Vì mô hình đã có nền tảng, học nhanh quá sẽ bị "tẩy não" quên mất kiến thức cũ (Catastrophic Forgetting)
    learning_rate=2e-5,               
    
    # Vẫn giữ nguyên cấu hình an toàn cho Windows & RTX 4060 Ti
    per_device_train_batch_size=64,   
    per_device_eval_batch_size=64,    
    gradient_accumulation_steps=1,    
    fp16=True,                       
    dataloader_num_workers=0,         
    group_by_length=True,             
    
    # CHIẾN THUẬT 2: Data ít và chất lượng cao, chỉ cần ép 3-5 epoch là đủ ngấm
    num_train_epochs=10,               
    warmup_ratio=0.1,
    weight_decay=0.01,
    
    logging_dir=f"./logs/{run_name}",
    logging_steps=10,                 # Data ít hơn nên hạ log step xuống để dễ nhìn đồ thị
    report_to="tensorboard",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=split_new_dataset["train"],
    eval_dataset=split_new_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] 
)

# 4. CHÍNH THỨC HỌC NHỒI
print(f"\n>>> ĐANG CHẠY TRIAL MỚI: {run_name}")
trainer.train()

Reusing TensorBoard on port 6006 (pid 45796), started 23:26:14 ago. (Use '!kill 45796' to kill it.)

c:\Users\ADMIN\anaconda3\envs\tracking-barbell-exercises\lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19744\664738723.py:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(



>>> ĐANG CHẠY TRIAL MỚI: Phase2_HardExamples_LR_2e-5


  0%|          | 0/580 [00:00<?, ?it/s]c:\Users\ADMIN\anaconda3\envs\tracking-barbell-exercises\lib\site-packages\transformers\models\mbart\modeling_mbart.py:495: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
  2%|▏         | 11/580 [00:15<03:48,  2.49it/s] 

{'loss': 0.6135, 'grad_norm': inf, 'learning_rate': 3.103448275862069e-06, 'epoch': 0.17}


...

 80%|████████  | 464/580 [04:20<00:44,  2.59it/s]Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
                                                 
 80%|████████  | 464/580 [04:21<00:44,  2.59it/s]

{'eval_loss': 0.32113412022590637, 'eval_runtime': 0.8815, 'eval_samples_per_second': 465.1, 'eval_steps_per_second': 7.941, 'epoch': 8.0}


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].
 80%|████████  | 464/580 [04:28<01:07,  1.73it/s]

{'train_runtime': 268.6695, 'train_samples_per_second': 137.343, 'train_steps_per_second': 2.159, 'train_loss': 0.2920930095273873, 'epoch': 8.0}


TrainOutput(global_step=464, training_loss=0.2920930095273873, metrics={'train_runtime': 268.6695, 'train_samples_per_second': 137.343, 'train_steps_per_second': 2.159, 'total_flos': 1526569381822464.0, 'train_loss': 0.2920930095273873, 'epoch': 8.0})

In [5]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# 1. Trỏ đúng đường dẫn tới checkpoint 464 (Dùng raw string 'r' để tránh lỗi path Windows)
best_model_path = r"C:\Code\TeenCodeTranslator\fine_tune\outputs\Phase2_HardExamples_LR_2e-5\checkpoint-464" 

print("Đang nạp bộ não thần đồng từ Checkpoint 464...")
tokenizer = AutoTokenizer.from_pretrained(best_model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(best_model_path)

# 2. Khởi tạo Pipeline inference trên GPU
translator = pipeline(
    "text2text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    device=0 if torch.cuda.is_available() else -1
)

# 3. Chạy test nghiệm thu với những ca khó nhất
test_cases = [
    "mai ik cf k duma",
    "hnao rảnh đi chs vs t kbt nma t thik m vcl =))",
    "tk đó đtrai nma cbi chtay r hsy 😂",
    "htrc t thấy n đi zới ngt r ksao đâu m"
]

print("\n" + "="*40)
print("🏆 KẾT QUẢ DỊCH CỦA CHECKPOINT 464 🏆")
print("="*40)

for text in test_cases:
    result = translator(
        text, 
        max_length=64,
        num_beams=5,             # Dùng 5 luồng suy nghĩ để chọn từ tốt nhất
        early_stopping=True
    )
    print(f"📝 Input  : {text}")
    print(f"✨ Output : {result[0]['generated_text']}")
    print("-" * 40)

Đang nạp bộ não thần đồng từ Checkpoint 464...

🏆 KẾT QUẢ DỊCH CỦA CHECKPOINT 464 🏆
📝 Input  : mai ik cf k duma
✨ Output : mai đi cà phê không duma
----------------------------------------
📝 Input  : hnao rảnh đi chs vs t kbt nma t thik m vcl =))
✨ Output : hôm nào rảnh đi chơi với tôi không biết nhưng mà tôi thích mày vcl =))
----------------------------------------
📝 Input  : tk đó đtrai nma cbi chtay r hsy 😂
✨ Output : thằng đó đẹp trai nhưng mà chuẩn bị chia tay rồi hay sao ý
----------------------------------------
📝 Input  : htrc t thấy n đi zới ngt r ksao đâu m
✨ Output : hôm trước tôi thấy nó đi với người ta rồi không sao đâu mày
----------------------------------------
